# H01 · Measuring temperature right - dividers, NTCs, ADC bits and oversampling

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/08_hardware_electronics/H01_measuring_temperature_right_dividers_ntcs_adc_bi.ipynb)

**Track:** Hardware and electronics · **Skill axis (sheet):** Hardware and electronics · **Time:** about 55 min

**Skill Hub concepts:** Voltage divider and thermistor Beta equation; Ratiometric measurement; ADC resolution and effective number of bits; Oversampling

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("H01")

## Map: from a resistor to a temperature you can trust

```
 Vref ──[ R_fixed ]──┬── ADC pin            V_node / V_ref = R_ntc / (R_fixed + R_ntc)
                     │                      code / 2^N     = same ratio   (V_ref cancels: ratiometric)
                  [ NTC ]                   R_ntc = R_fixed * code / (2^N - code)
                     │
                    GND
```

| Idea | Formula / rule |
|---|---|
| Divider | $V_{out} = V_{in}\,\dfrac{R_{bottom}}{R_{top} + R_{bottom}}$ |
| NTC Beta model | $\dfrac{1}{T} = \dfrac{1}{T_0} + \dfrac{1}{\beta}\ln\dfrac{R}{R_0}$ (kelvin; $R_0$ at $T_0 = 298.15$ K) |
| Ratiometric | divider and ADC share the same reference, so reference drift and noise cancel |
| Ideal ADC SNR | $6.02N + 1.76$ dB for a full-scale sine |
| Effective number of bits | $\text{ENOB} = (\text{SINAD} - 1.76)/6.02$ from a measured sine |
| Oversampling | average $4^k$ samples (with at least 1 LSB of noise as dither) and shift right by $k$: $k$ more bits, $4^k$ times slower |

In [ ]:
import math
import numpy as np

## Task 1: divider, ratiometric reading and the Beta equation

In [ ]:
def divider(v_in, r_top, r_bottom):
    raise NotImplementedError  # TODO

def ntc_ohms_from_code(code, bits, r_fixed):
    raise NotImplementedError  # TODO

def ntc_temp_c(r, r0=10e3, t0_c=25.0, beta=3950.0):
    raise NotImplementedError  # TODO

lab.check("ntc", divider, ntc_ohms_from_code, ntc_temp_c)

In [ ]:
%%writefile h01_ntc.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/h01.h
// Grade it with lab.check("ntc", "h01_ntc.c"). For C++, name the file .cpp in both places.
#include "h01.h"

double divider(double v_in, double r_top, double r_bottom) { (void)v_in; (void)r_top; (void)r_bottom; return 0; /* TODO */ }
double ntc_ohms_from_code(uint32_t code, int bits, double r_fixed) { (void)code; (void)bits; (void)r_fixed; return 0; /* TODO */ }
double ntc_temp_c(double r, double r0, double t0_c, double beta) { (void)r; (void)r0; (void)t0_c; (void)beta; return 0; /* TODO */ }

In [ ]:
# lab.check("ntc", "h01_ntc.c")  # uncomment after writing the C/C++ cell above

## Task 2: measure ENOB from a captured sine
Fit $a\sin(2\pi f_0 t) + b\cos(2\pi f_0 t) + c$ by least squares (the frequency is known), then
SINAD $= 20\log_{10}(\text{RMS of the fitted sine} / \text{RMS of the residual})$.

In [ ]:
def enob(samples, fs, f0):
    raise NotImplementedError  # TODO

lab.check("enob", enob)

In [ ]:
%%writefile h01_enob.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/h01.h
// Grade it with lab.check("enob", "h01_enob.c"). For C++, name the file .cpp in both places.
#include "h01.h"

double enob(const double *samples, size_t n, double fs, double f0) {
    (void)samples; (void)n; (void)fs; (void)f0;
    return 0; /* TODO: fit a sin + b cos + c (3x3 normal equations), noise = residual RMS, SINAD -> ENOB */
}

In [ ]:
# lab.check("enob", "h01_enob.c")  # uncomment after writing the C/C++ cell above

## Task 3: oversample and decimate
Sum consecutive groups of $4^k$ samples, add $2^{k-1}$ (round to nearest) and shift right by $k$ (integer math, as
firmware would). Drop an incomplete last group.

In [ ]:
def oversample(samples, extra_bits):
    raise NotImplementedError  # TODO

lab.check("oversample", oversample)

In [ ]:
%%writefile h01_oversample.c
// C/C++ version of Task 3 (optional, same tests). Declarations: roboprep/c/include/h01.h
// Grade it with lab.check("oversample", "h01_oversample.c"). For C++, name the file .cpp in both places.
#include "h01.h"

size_t oversample(const int32_t *samples, size_t n, int extra_bits, int32_t *out, size_t out_cap) {
    (void)samples; (void)n; (void)extra_bits; (void)out; (void)out_cap;
    return 0; /* TODO */
}

In [ ]:
# lab.check("oversample", "h01_oversample.c")  # uncomment after writing the C/C++ cell above

## Checkpoint: Math
**M1 (Apply).** 3.3 V into a divider with 10 kOhm on top and 4.7 kOhm on the bottom. Output volts (3 decimals)?

**M2 (Apply).** 10 kOhm NTC ($\beta = 3950$, $R_0$ at 25 C) now reads 5 kOhm. Temperature in Celsius (2 decimals)?

**M3 (Apply).** Samples to average for 2 extra bits?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Understand).** The 3.3 V regulator drifts 2% with temperature. Why does the NTC reading barely change?
- a) the NTC compensates
- b) the divider and the ADC reference come from the same supply, so the ratio code / 2^N is independent of it (ratiometric)
- c) the ADC has an internal reference

**K2 (Evaluate).** Your 12-bit ADC measures ENOB = 9.5 on the board. List likely causes and fixes, and when
oversampling helps or does not.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `h01.h`).
3. In `roboprep/checks/h01.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()